# SpamShield — Exploratory Analysis & Pipeline Walkthrough

**COMP11117 · MSc Information Technology · University of the West of Scotland**  
**Student:** Favour Nnenna Ogbonnaya-John &nbsp;|&nbsp; **Supervisor:** Dr. Nurudeen Oladehinbo Salau

---

This notebook documents the full analytical journey behind the SpamShield project:

1. Dataset loading and overview  
2. Exploratory Data Analysis (EDA)  
3. Text preprocessing walkthrough  
4. TF-IDF feature extraction  
5. Class imbalance handling with SMOTE  
6. Model training (Logistic Regression, SVM, XGBoost)  
7. Evaluation metrics and visualisations  
8. Sample predictions  

> **Run from the `SpamShield/` root** or keep the path setup cell below as-is so the `src/` modules resolve correctly.

## 0. Environment Setup

In [ ]:
import sys, os

# Ensure the SpamShield root is on the path regardless of where the notebook is launched from
ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('All imports successful.')

---
## 1. Dataset Loading & Overview

The UCI SMS Spam Collection contains 5,574 English SMS messages hand-labelled as **spam** or **ham** (legitimate).  
Source: Almeida and Hidalgo (2011).

In [ ]:
DATA_PATH = 'data/spam.csv'

if not os.path.exists(DATA_PATH):
    print('Dataset not found — downloading...')
    from download_data import download
    download()

df = pd.read_csv(DATA_PATH, encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'message']

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
print('=== Basic Statistics ===')
print(f"Total messages : {len(df):,}")
print(f"Spam messages  : {(df['label']=='spam').sum():,}  ({(df['label']=='spam').mean()*100:.1f}%)")
print(f"Ham messages   : {(df['label']=='ham').sum():,}  ({(df['label']=='ham').mean()*100:.1f}%)")
print(f"Duplicate rows : {df.duplicated().sum()}")
print(f"Null values    : {df.isnull().sum().sum()}")

---
## 2. Exploratory Data Analysis

### 2.1 Class Distribution

The dataset is heavily imbalanced — roughly **87% ham** and **13% spam**. This matters because a naive classifier that always predicts ham would achieve ~87% accuracy without learning anything useful. This is why F1-score and ROC-AUC are used as primary metrics rather than accuracy.

In [ ]:
counts = df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colours = ['#3d7cf4', '#f0384f']
axes[0].bar(counts.index, counts.values, color=colours, edgecolor='white', linewidth=0.8)
axes[0].set_title('Message Class Counts', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for i, (label, v) in enumerate(counts.items()):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=11)

# Pie chart
axes[1].pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=colours,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('UCI SMS Spam Collection — Class Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nClass imbalance ratio (ham:spam) = {counts["ham"]/counts["spam"]:.1f}:1')

### 2.2 Message Length Analysis

Spam messages tend to be longer than ham — they often include promotional text, phone numbers, and URLs.

In [ ]:
df['char_count'] = df['message'].str.len()
df['word_count'] = df['message'].str.split().str.len()

print('=== Character Count by Class ===')
print(df.groupby('label')['char_count'].describe().round(1).to_string())
print()
print('=== Word Count by Class ===')
print(df.groupby('label')['word_count'].describe().round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, colour in [('ham', '#3d7cf4'), ('spam', '#f0384f')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['char_count'], bins=50, alpha=0.6, label=label, color=colour)
    axes[1].hist(subset['word_count'], bins=40, alpha=0.6, label=label, color=colour)

axes[0].set_title('Character Count Distribution', fontweight='bold')
axes[0].set_xlabel('Characters per message')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].set_title('Word Count Distribution', fontweight='bold')
axes[1].set_xlabel('Words per message')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.3 Most Frequent Words — Spam vs Ham

Raw word frequencies (before preprocessing) reveal the vocabulary gap between spam and legitimate messages.

In [ ]:
import re

def raw_tokens(series):
    """Simple whitespace tokeniser on lowercased text — for raw frequency inspection only."""
    all_words = []
    for msg in series:
        tokens = re.findall(r'[a-z]+', msg.lower())
        all_words.extend(tokens)
    return Counter(all_words)

spam_counts = raw_tokens(df[df['label'] == 'spam']['message'])
ham_counts  = raw_tokens(df[df['label'] == 'ham']['message'])

# Remove very common stopwords for a more meaningful display
SKIP = {'to', 'a', 'i', 'you', 'the', 'and', 'is', 'in', 'it',
        'of', 'my', 'me', 'for', 'have', 'on', 'are', 'this',
        'we', 'ur', 'u', 'or', 'be', 'will', 'do', 'so', 'but',
        'not', 'if', 'at', 'he', 'she', 'that', 'with', 'your',
        'can', 'from', 'just', 'get', 'an', 'up', 'no', 'was',
        'all', 'now', 'how', 'go', 'its', 'by', 'out', 'as',
        'call', 'ok', 'hi', 'hey', 'im', 'am', 'has', 'had'}

top_spam = [(w, c) for w, c in spam_counts.most_common(200) if w not in SKIP][:15]
top_ham  = [(w, c) for w, c in ham_counts.most_common(200)  if w not in SKIP][:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([w for w, _ in reversed(top_spam)], [c for _, c in reversed(top_spam)],
             color='#f0384f', edgecolor='white')
axes[0].set_title('Top 15 Words in SPAM Messages', fontweight='bold')
axes[0].set_xlabel('Raw frequency')

axes[1].barh([w for w, _ in reversed(top_ham)], [c for _, c in reversed(top_ham)],
             color='#3d7cf4', edgecolor='white')
axes[1].set_title('Top 15 Words in HAM Messages', fontweight='bold')
axes[1].set_xlabel('Raw frequency')

plt.tight_layout()
plt.show()

### 2.4 Special Character Patterns in Spam

Spam messages frequently contain URLs, phone numbers, and excessive punctuation — all of which the preprocessing pipeline strips.

In [ ]:
def has_url(text):   return bool(re.search(r'http\S+|www\.\S+', text, re.I))
def has_phone(text): return bool(re.search(r'\+?\d[\d\s\-().]{7,}\d', text))
def has_caps(text):  return sum(1 for c in text if c.isupper()) / max(len(text), 1)

for label in ['ham', 'spam']:
    subset = df[df['label'] == label]['message']
    url_pct   = subset.apply(has_url).mean() * 100
    phone_pct = subset.apply(has_phone).mean() * 100
    caps_avg  = subset.apply(has_caps).mean() * 100
    print(f"{label.upper():5s} — URL: {url_pct:5.1f}%  |  Phone number: {phone_pct:5.1f}%  |  Avg CAPS ratio: {caps_avg:4.1f}%")

---
## 3. Text Preprocessing Walkthrough

The `clean_text` function in `src/preprocess.py` applies a seven-step pipeline. The cell below demonstrates each step on a real spam message.

In [ ]:
import nltk
nltk.download('punkt',      quiet=True)
nltk.download('punkt_tab',  quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('wordnet',    quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

EXAMPLE_SPAM = "WINNER!! As a valued network customer you have been selected to receive a £900 prize reward! Call 09061701461 from land line. Claim code KL341. Valid 12 hours only."
EXAMPLE_HAM  = "Hi, I'm running about 20 minutes late. Will be there soon. Sorry!"

def preprocess_steps(text):
    steps = {'0. Original': text}

    # Step 1: Lowercase
    text = text.lower()
    steps['1. Lowercase'] = text

    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    steps['2. Remove URLs'] = text

    # Step 3: Remove email-like strings
    text = re.sub(r'\S+@\S+', ' ', text)
    steps['3. Remove emails'] = text

    # Step 4: Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-().]{7,}\d', ' ', text)
    steps['4. Remove phones'] = text

    # Step 5: Remove punctuation / non-alpha
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    steps['5. Strip punctuation'] = text

    # Step 6: Tokenise + remove stopwords
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    steps['6. Tokenise + stopwords'] = str(tokens)

    # Step 7: Lemmatise
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    steps['7. Lemmatise (final)'] = ' '.join(tokens)

    return steps

print('=== SPAM MESSAGE — step-by-step ===')
for step, result in preprocess_steps(EXAMPLE_SPAM).items():
    print(f'  {step:<30s}: {result[:100]}')

print()
print('=== HAM MESSAGE — step-by-step ===')
for step, result in preprocess_steps(EXAMPLE_HAM).items():
    print(f'  {step:<30s}: {result[:100]}')

In [ ]:
# Apply the full preprocessing pipeline (using the actual src module)
from src.preprocess import preprocess_series

print('Preprocessing all messages...')
df['clean'] = preprocess_series(df['message'])

print(f'Done. Sample cleaned messages:')
df[['label', 'message', 'clean']].sample(5, random_state=1)

In [ ]:
# How much does cleaning compress message length?
df['clean_word_count'] = df['clean'].str.split().str.len()

reduction = (1 - df['clean_word_count'] / df['word_count']).mean() * 100
print(f'Average vocabulary reduction after preprocessing: {reduction:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
for label, colour in [('ham', '#3d7cf4'), ('spam', '#f0384f')]:
    subset = df[df['label'] == label]
    ax.hist(subset['clean_word_count'], bins=40, alpha=0.6, label=label, color=colour)

ax.set_title('Word Count After Preprocessing', fontweight='bold')
ax.set_xlabel('Words per cleaned message')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Feature Extraction — TF-IDF

TF-IDF (Term Frequency–Inverse Document Frequency) converts cleaned text into a numerical matrix.  
- **TF** rewards terms that appear frequently in a document.  
- **IDF** down-weights terms that appear in nearly every document (they carry less discriminative power).  
- The vectoriser uses **unigrams + bigrams** (`ngram_range=(1,2)`) and a vocabulary of **5,000 features**.

In [ ]:
from sklearn.model_selection import train_test_split
from src.features import build_features

df['label_int'] = (df['label'] == 'spam').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label_int'],
    test_size=0.2, random_state=42, stratify=df['label_int']
)

print(f'Train size: {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'Train spam rate: {y_train.mean()*100:.1f}%  |  Test spam rate: {y_test.mean()*100:.1f}%')

In [ ]:
print('Building TF-IDF features and applying SMOTE...')
X_train_res, y_train_res, X_test_tfidf, vectorizer = build_features(X_train, X_test, y_train)

print(f'\nTF-IDF matrix shapes:')
print(f'  Training (before SMOTE): {X_train.shape[0]:,} messages')
print(f'  Training (after SMOTE) : {X_train_res.shape[0]:,} samples  ← resampled to balance classes')
print(f'  Test                   : {X_test_tfidf.shape[0]:,} messages')
print(f'  Vocabulary (features)  : {X_test_tfidf.shape[1]:,}')

In [ ]:
# Inspect top TF-IDF features for spam vs ham
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())

# Mean TF-IDF score per feature for each class
X_train_tfidf_orig = vectorizer.transform(X_train)
spam_mask = y_train.values == 1
ham_mask  = y_train.values == 0

spam_mean = np.asarray(X_train_tfidf_orig[spam_mask].mean(axis=0)).flatten()
ham_mean  = np.asarray(X_train_tfidf_orig[ham_mask].mean(axis=0)).flatten()

top_n = 15
top_spam_idx = spam_mean.argsort()[-top_n:][::-1]
top_ham_idx  = ham_mean.argsort()[-top_n:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names[top_spam_idx][::-1], spam_mean[top_spam_idx][::-1],
             color='#f0384f', edgecolor='white')
axes[0].set_title('Top 15 TF-IDF Features — SPAM', fontweight='bold')
axes[0].set_xlabel('Mean TF-IDF score')

axes[1].barh(feature_names[top_ham_idx][::-1], ham_mean[top_ham_idx][::-1],
             color='#3d7cf4', edgecolor='white')
axes[1].set_title('Top 15 TF-IDF Features — HAM', fontweight='bold')
axes[1].set_xlabel('Mean TF-IDF score')

plt.tight_layout()
plt.show()

---
## 5. Class Imbalance — SMOTE

The dataset has an **87:13 ham:spam imbalance**. Without correction, models bias toward predicting ham.  
SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic spam samples in feature space — applied **only to the training split** to prevent data leakage into the test set.

In [ ]:
before_counts = pd.Series(y_train).value_counts().sort_index()
after_counts  = pd.Series(y_train_res).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colours = ['#3d7cf4', '#f0384f']
labels  = ['Ham (0)', 'Spam (1)']

axes[0].bar(labels, before_counts.values, color=colours, edgecolor='white')
axes[0].set_title('Training Set — Before SMOTE', fontweight='bold')
axes[0].set_ylabel('Sample count')
for i, v in enumerate(before_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center')

axes[1].bar(labels, after_counts.values, color=colours, edgecolor='white')
axes[1].set_title('Training Set — After SMOTE', fontweight='bold')
axes[1].set_ylabel('Sample count')
for i, v in enumerate(after_counts.values):
    axes[1].text(i, v + 20, str(v), ha='center')

plt.suptitle('Effect of SMOTE on Class Distribution', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Before SMOTE — ham: {before_counts[0]:,}  spam: {before_counts[1]:,}  ratio: {before_counts[0]/before_counts[1]:.1f}:1')
print(f'After SMOTE  — ham: {after_counts[0]:,}  spam: {after_counts[1]:,}  ratio: {after_counts[0]/after_counts[1]:.1f}:1')

---
## 6. Model Training

Three classifiers are trained on the SMOTE-balanced TF-IDF features:

| Classifier | Configuration | Rationale |
|---|---|---|
| **Logistic Regression** | `max_iter=1000, random_state=42` | Linear probabilistic baseline; interpretable weights |
| **SVM (LinearSVC)** | `max_iter=2000, random_state=42` | Strong margin-based classifier for text; fast on sparse data |
| **XGBoost** | `n_estimators=200, eval_metric='logloss'` | Gradient-boosted ensemble; captures non-linear patterns |

In [ ]:
from src.train import train_all

print('Training all three classifiers on SMOTE-balanced training data...')
trained = train_all(X_train_res, y_train_res)
print('\nAll classifiers trained and saved to models/.')

---
## 7. Evaluation

Each model is evaluated on the held-out test set using six metrics:  
**Accuracy · Precision · Recall · F1-score · Confusion Matrix · ROC-AUC**

F1 and ROC-AUC are the primary comparison metrics because they are robust to class imbalance.

In [ ]:
from src.evaluate import evaluate_all, save_plots, save_metrics_csv, best_model_name

print('Evaluating on held-out test set...')
results = evaluate_all(trained, X_test_tfidf, y_test)

In [ ]:
# Display as a formatted table
rows = []
for r in results:
    rows.append({
        'Model': r['name'].replace('_', ' ').title(),
        'Accuracy': round(r['accuracy'], 4),
        'Precision': round(r['precision'], 4),
        'Recall': round(r['recall'], 4),
        'F1': round(r['f1'], 4),
        'ROC-AUC': round(r['roc_auc'], 4),
    })

metrics_df = pd.DataFrame(rows)

def highlight_max(s):
    return ['background-color: #d4edda; font-weight: bold' if v == s.max() else '' for v in s]

styled = (metrics_df.style
          .apply(highlight_max, subset=['Accuracy','Precision','Recall','F1','ROC-AUC'])
          .set_caption('Table 4.1 — Model performance on held-out test set (80/20 stratified split · 5,574 messages)'))
styled

In [ ]:
# ── Confusion matrices ───────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, r in zip(axes, results):
    sns.heatmap(
        r['confusion_matrix'], annot=True, fmt='d',
        cmap='Blues', ax=ax,
        xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'],
        cbar=False
    )
    ax.set_title(r['name'].replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Held-out Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC curves ──────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve

colours = ['#3d7cf4', '#f0384f', '#00bf85']

plt.figure(figsize=(7, 5))
for r, colour in zip(results, colours):
    fpr, tpr, _ = roc_curve(y_test, r['y_score'])
    plt.plot(fpr, tpr, label=f"{r['name'].replace('_',' ').title()} (AUC={r['roc_auc']:.4f})",
             color=colour, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Classifiers', fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Grouped bar chart: all metrics side-by-side ──────────────────────────────
metrics_list = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
x = np.arange(len(metrics_list))
width = 0.25
bar_colours = ['#3d7cf4', '#f0384f', '#00bf85']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (r, colour) in enumerate(zip(results, bar_colours)):
    values = [r[m] for m in metrics_list]
    offset = (i - len(results) / 2 + 0.5) * width
    bars = ax.bar(x + offset, values, width, label=r['name'].replace('_', ' ').title(),
                  color=colour, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').upper() for m in metrics_list])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 7.1 Results Interpretation

| Model | Best metric | Interpretation |
|---|---|---|
| **XGBoost** | F1 (0.9091), Precision (0.9489) | Strongest primary metric; fewest false positives |
| **Logistic Regression** | ROC-AUC (0.9875) | Best class separation at the probabilistic level; preferred when recall matters |
| **SVM** | Competitive recall (0.9195) | Weakest precision; threshold not well-calibrated on this dataset |

**Why accuracy misleads here:** A classifier that always predicts ham achieves ~87% accuracy. All three models score above 94% accuracy — but that uplift over the naive baseline is only meaningful when precision and recall are also strong, which is why F1 is the primary comparison metric.

In [ ]:
best = best_model_name(results)
print(f'Best model by F1-score: {best}')
save_plots(results, y_test)
save_metrics_csv(results)
print('Reports saved to reports/')

---
## 8. Sample Predictions

Testing the deployed inference path (`src/predict.py`) on hand-crafted examples — including ambiguous messages that sit near the decision boundary.

In [ ]:
from src.predict import predict

test_messages = [
    # Clear spam
    ("Congratulations! You have been selected as a winner. Claim your free prize by calling 07900 123456 now.", 'Spam'),
    ("Ur acc has been selected 4 a special reward. Reply 2 claim.", 'Spam'),
    ("FREE entry in 2 a wkly comp to win FA Cup final tkts. Text FA to 87121", 'Spam'),
    # Clear ham
    ("Hi, I'm running about 20 minutes late. Will be there soon.", 'Ham'),
    ("Can we reschedule our meeting to Thursday instead?", 'Ham'),
    ("Thanks for dinner last night, it was really lovely.", 'Ham'),
    # Ambiguous / borderline
    ("You have a new voicemail. Please call 123 to retrieve it.", '?'),
    ("Your order has been dispatched. Track at www.example.com", '?'),
]

print(f"{'Message':<65} {'Expected':<8} {'Prediction':<10} {'Certainty':>9}")
print('-' * 95)

for model_name in ['logistic_regression', 'svm', 'xgboost']:
    print(f'\n  Model: {model_name}')
    for msg, expected in test_messages:
        verdict, certainty = predict(msg, model_name)
        cert_str = f'{certainty:.1%}' if certainty else 'n/a'
        flag = '✓' if verdict == expected or expected == '?' else '✗'
        print(f"  {flag} {msg[:62]:<63} {expected:<8} {verdict:<10} {cert_str:>9}")

---
## 9. Logistic Regression — Feature Weight Inspection

Because Logistic Regression is a linear model, its coefficients directly show which terms push a message toward spam (positive weight) or ham (negative weight). This interpretability is one of its key advantages.

In [ ]:
from src.train import load_model

lr = load_model('logistic_regression')
feature_names = vectorizer.get_feature_names_out()
coefs = lr.coef_[0]

top_n = 20
top_spam_idx = coefs.argsort()[-top_n:][::-1]
top_ham_idx  = coefs.argsort()[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(feature_names[top_spam_idx][::-1], coefs[top_spam_idx][::-1],
             color='#f0384f', edgecolor='white')
axes[0].set_title('Top 20 Spam Indicators\n(highest positive coefficients)', fontweight='bold')
axes[0].set_xlabel('Logistic Regression coefficient')

axes[1].barh(feature_names[top_ham_idx][::-1], coefs[top_ham_idx][::-1],
             color='#3d7cf4', edgecolor='white')
axes[1].set_title('Top 20 Ham Indicators\n(most negative coefficients)', fontweight='bold')
axes[1].set_xlabel('Logistic Regression coefficient')

plt.tight_layout()
plt.show()

---
## 10. Summary

| Finding | Detail |
|---|---|
| **Dataset** | 5,574 SMS messages; 87.4% ham, 12.6% spam |
| **Preprocessing** | 7-step pipeline; ~40–50% vocabulary reduction |
| **Features** | TF-IDF, unigrams+bigrams, 5,000 features |
| **Imbalance** | SMOTE equalises training split only |
| **Best model** | XGBoost (F1=0.9091, Precision=0.9489) |
| **Best separator** | Logistic Regression (ROC-AUC=0.9875) |
| **Key takeaway** | Classical ML with careful design achieves >97% accuracy, >0.90 F1, and >0.98 AUC on this task |

### Recommended future work

- **Transformer comparison** — fine-tune DistilBERT on the same split under identical evaluation conditions
- **SVM threshold calibration** — `CalibratedClassifierCV` wrapper to improve precision without sacrificing recall  
- **Email corpus** — replicate the pipeline on a true email spam dataset to validate title–dataset alignment  
- **Cross-lingual extension** — test on non-English SMS datasets